# Does AML-specific adaptation fix foundation-model false positives after treatment?

**Question.** Does continued training of scGPT on AML cells reduce *false-positive malignant calls among genotype-defined non-malignant progenitors after induction chemotherapy*, at a fixed detection sensitivity, compared with the original scGPT, HVG-PCA, scVI, and the same architecture trained from scratch?

**Data.** Naldini et al., *Nat Commun* 2023 ([GSE185991](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE185991)): 10x 3′ scRNA-seq of NPM1-mutated AML marrow at diagnosis (DX), day 14 / day 30 after induction (post) and relapse (REL), with a **per-cell genotype call** from NPM1 reads (`MUT` = mutant reads, `WT` = wild-type reads only; the rest unlabelled). 10 patients have deposited count matrices. The authors describe this call as what lets them separate leukemia from regenerating normal hematopoiesis after chemotherapy, which is the distinction this study needs.

## Design (fixed before any result)
| Element | Choice |
|---|---|
| Evaluation unit | **patient** (patient-grouped K-fold; adaptation re-run inside each fold on *training patients only*, unlabelled) |
| Task | cell-level MUT vs WT among progenitor-compartment cells (SingleR GMP/CMP/MEP/HSC) |
| Probe | identical regularised logistic regression on every representation; C by inner patient-grouped CV |
| **Stage-confound control** | malignant cells are mostly diagnosis, normal cells mostly post-treatment, so every stage × label group is weighted equally (a probe cannot score "post-treatment" instead of "leukemic") |
| Operating point | threshold set on *training* patients for 90% sensitivity; never tuned on the test fold |
| **Primary endpoint** | false-positive rate in **day 14/30 WT progenitors**, patient-macro-averaged, 95% patient-cluster bootstrap CI |
| Secondary | MUT-vs-WT AUROC within post-treatment progenitors |
| Contrasts | C1 adapted vs original scGPT · C2 vs PCA · C3 vs scVI · C4 vs from-scratch |
| Pass rule | CI of the FPR difference entirely below 0 **and** sensitivity within 5 points of the comparator |
| Abandon rule | if C1–C3 do not all pass, drop the model-development claim |

## What the data check showed (computed before building this)
* Post-treatment WT progenitors: ~2,000 cells; ~8 patients have ≥10 each (largest patient ~1/3). Post-treatment MUT progenitors: ~700 cells, but only ~4 patients with ≥10. Cell 4 recomputes all of this.
* Label noise: at diagnosis the WT-only fraction among genotype-called cells is ≈0% in most patients, so a WT call after treatment is rarely a leukemic cell that dropped the mutant allele. It is **not** small in every patient (PT06, PT07 hold real residual normal cells at diagnosis), and the cell 4 table shows it per patient.

## Limits that stay
* **NPM1-mutated AML only, one locus, one centre.** A positive result would not transfer to other genotypes without replication.
* Progenitor compartment is a coarse SingleR label, not a curated gate. Day 14/30 samples include persistent disease. No healthy-donor stratum (those matrices are not deposited).
* Adaptation is masked-value prediction (GEP + GEPC, mask ratios 0.25/0.5/0.75); the original scGPT also used a generative objective.
* Not a clinical MRD assay; nothing here estimates MRD sensitivity.
* Whether the adult/paediatric status of every patient fits your adult-only constraint should be confirmed from the paper's Supplementary Fig. 1B; I could not verify it from the text.

**Runtime (T4):** ~10–15 min per adaptation × 4 folds × (adapted + scratch) plus scVI ≈ 1.5–2.5 h. `RUN_SCRATCH=False` roughly halves it. `SMOKE_TEST=True` in cell 2 runs a synthetic 2–3 minute plumbing check. Caches and results live on Google Drive and resume after disconnects.


In [ ]:
#@title 1 · Install prerequisites (Colab)
# Same pinned stack that ran your zero-shot notebook successfully.
%pip -q install "anndata>=0.10" "scanpy>=1.10" "huggingface_hub>=0.25" "transformers>=4.40,<4.49" "scgpt==0.2.4" "scikit-learn" "pandas" "matplotlib" "seaborn" "tqdm" "requests" "pyreadr"

In [ ]:
#@title 2 · Configuration (everything tunable lives here)
import os, sys, re, io, json, math, time, gzip, shutil, warnings, itertools
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
import torch
from tqdm.auto import tqdm
from IPython.display import display
warnings.filterwarnings('ignore')

SEED = 42
SMOKE_TEST = False       # True = synthetic data + tiny random "pretrained" model: a 2-3 min plumbing check, NOT science
USE_DRIVE  = True        # keep downloads/caches on Google Drive so a runtime disconnect doesn't lose work
IN_COLAB   = 'google.colab' in sys.modules
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- design (pre-specified; change BEFORE looking at results, not after) ----
N_FOLDS         = 4      # patient-grouped outer folds (10 patients); adaptation is re-run inside each fold on TRAINING patients only
TARGET_TPR      = 0.90   # fixed detection sensitivity; threshold is set on training patients, never on the test fold
PROG_HPCA = {'GMP', 'CMP', 'MEP', 'HSC_CD34+', 'HSC_-G-CSF', 'BM & Prog.'}   # SingleR/HPCA labels shipped with the metadata (coarse!)
STAGE_OF = {'DX': 'dx', 'D14': 'post', 'D30': 'post', 'REL': 'relapse', 'HD': 'healthy'}   # post = day 14/30 after induction
MIN_CELLS_PATIENT_STRATUM = 10   # a patient contributes to a stratum's macro-average only with >= this many cells
N_BOOT = 5000            # patient-cluster bootstrap replicates
C_GRID = (0.01, 0.1, 1.0)        # logistic-probe regularisation, chosen by INNER patient-grouped CV
PROBE_MAX_PER_SAMPLE_LABEL = 300 # stops one huge sample (e.g. AML328-D29) dominating probe training

# ---- scGPT adaptation ----
MAX_LEN        = 1200    # same as your zero-shot run
ADAPT_STEPS    = 1500    # optimizer steps per fold (~1 pass over a fold's training cells at batch 16)
ADAPT_BATCH    = 8       # micro-batch; lower if a T4 runs out of memory
ADAPT_ACCUM    = 2       # effective batch = 16
ADAPT_LR       = 2e-5    # continued training of pretrained weights (original pretraining used 1e-4)
SCRATCH_LR     = 1e-4    # same architecture, random init
MASK_RATIOS    = (0.25, 0.5, 0.75)  # original scGPT pretraining mask ratios
MAX_ADAPT_CELLS_PER_SAMPLE = 1500
RUN_SCRATCH    = True    # ablation: same architecture + same AML-only objective, NO foundation-model pretraining
RUN_SCVI       = True    # conventional baseline #2 (skipped gracefully if scvi-tools is unusable)
TRAIN_FINAL_MODEL = True # last cell: adapt on ALL patients and save a checkpoint for the seven-patient follow-up notebook

if SMOKE_TEST:
    N_FOLDS, ADAPT_STEPS, ADAPT_BATCH, ADAPT_ACCUM, N_BOOT = 3, 30, 4, 1, 300
    MAX_LEN, MIN_CELLS_PATIENT_STRATUM, TRAIN_FINAL_MODEL = 200, 3, False

if IN_COLAB and USE_DRIVE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AML_FM_adaptation')
elif IN_COLAB:
    ROOT = Path('/content/AML_FM_adaptation')
else:
    ROOT = Path.cwd()/'AML_FM_adaptation'
if SMOKE_TEST: ROOT = ROOT.parent/'AML_FM_adaptation_SMOKE'
DATA, OUT, CACHE, MODEL = ROOT/'data', ROOT/'results', ROOT/'cache', ROOT/'models'
FIG = OUT/'figures'
for d in (DATA, OUT, CACHE, MODEL, FIG): d.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 0 if SMOKE_TEST else min(2, os.cpu_count() or 1)
print('device:', DEVICE, '| root:', ROOT, '| smoke:', SMOKE_TEST)
if DEVICE == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
else: print('WARNING: no GPU. Adaptation will be impractically slow; use Runtime > Change runtime type > T4/L4.')

In [ ]:
#@title 3 · Download Naldini et al. 2023 (GSE185991): per-cell NPM1 genotype metadata + 10x count matrices
import requests, urllib.request, urllib.error
GEO_TEXT = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE185991&targ=all&form=text&view=brief'
META_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE185nnn/GSE185991/suppl/GSE185991_Full_Patient_metadata.rds.gz'
MAN = DATA/'GSE185991_manifest.json'

def fetch(url, dest, tries=12):
    """Resumable download (HTTP Range) with backoff; verifies the gzip stream before saving."""
    if dest.exists(): return True
    time.sleep(0.5)                                                      # stay well under NCBI's request-rate limit
    buf, total = bytearray(), None
    for a in range(1, tries+1):
        try:
            req = urllib.request.Request(url, headers={'Range': f'bytes={len(buf)}-'} if buf else {})
            with urllib.request.urlopen(req, timeout=120) as r:
                if buf and r.status != 206: buf = bytearray()          # server ignored Range -> restart cleanly
                total = total or (int(r.headers['Content-Length']) + len(buf) if r.headers.get('Content-Length') else None)
                while True:
                    c = r.read(1 << 18)
                    if not c: break
                    buf += c
            if total and len(buf) != total: raise IOError('short read')
            gzip.decompress(bytes(buf))                                  # integrity check: raises on truncated gzip
            dest.write_bytes(bytes(buf)); return True
        except urllib.error.HTTPError as e:
            if e.code in (401, 403, 404):                                 # permanent: do not hammer the server
                print(f'  PERMANENT HTTP {e.code} for {dest.name}: skipping'); return False
            print(f'  retry {a}/{tries} {dest.name}: HTTP {e.code}'); time.sleep(min(60, 4*a))
        except Exception as e:
            print(f'  retry {a}/{tries} {dest.name}: {type(e).__name__} (have {len(buf)} bytes)'); time.sleep(min(60, 4*a))
    print(f'  gave up on {dest.name}'); return False

if SMOKE_TEST:
    print('SMOKE_TEST: skipping download'); manifest = {}
else:
    if not MAN.exists():
        r = requests.get(GEO_TEXT, timeout=180); r.raise_for_status(); man = {}
        for line in r.text.splitlines():
            if re.match(r'!Sample_supplementary_file_\d+ =', line):
                url = line.split('=', 1)[1].strip().replace('ftp://', 'https://')
                m = re.search(r'GSM\d+_(M\d+)_(barcodes\.tsv|features\.tsv|matrix\.mtx)\.gz$', url)
                if m: man.setdefault(m.group(1), {})[m.group(2)] = url
        MAN.write_text(json.dumps({k: v for k, v in man.items() if len(v) == 3}))
    manifest = json.loads(MAN.read_text())
    assert len(manifest) >= 33, f'unexpected manifest size {len(manifest)}'
    fetch(META_URL, DATA/'Full_Patient_metadata.rds.gz')
    (DATA/'mtx').mkdir(exist_ok=True)
    ok_samples = []
    for s, d in tqdm(sorted(manifest.items()), desc='10x samples'):
        if all([fetch(url, DATA/'mtx'/f'{s}_{part}.gz') for part, url in d.items()]): ok_samples.append(s)
    missing = sorted(set(manifest) - set(ok_samples))
    manifest = {s: manifest[s] for s in ok_samples}
    print(len(manifest), 'samples with matrices ready;', 'UNAVAILABLE (dropped, rerun later to retry):', missing)
    assert len(manifest) >= 28, 'too many samples failed to download; check connectivity and rerun'

In [ ]:
#@title 4 · Per-cell genotype labels, stage-confound check, and the FEASIBILITY GATE
def make_synthetic_meta(seed=0):
    r = np.random.default_rng(seed); rows = []
    for pi in range(1, 9):
        for tp, n in [('DX', 150), ('D14', 130), ('REL', 60)]:
            for j in range(n):
                pm = {'DX': (.85, .03), 'D14': (.3, .5), 'REL': (.8, .05)}[tp]; u = r.random()
                cl = 'MUT' if u < pm[0] else ('WT' if u < pm[0] + pm[1] else 'ND')
                rows.append(dict(cell=f'S{pi}{tp}_{j}', **{'orig.ident': f'S{pi}{tp}', 'PatientID': f'PT{pi:02d}', 'Timepoint': tp,
                                 'Classification': cl, 'main_hpca': r.choice(['GMP', 'CMP', 'HSC_CD34+', 'NK_cell', 'Monocyte'])}))
    return pd.DataFrame(rows).set_index('cell')

if SMOKE_TEST:
    META = make_synthetic_meta()
else:
    import pyreadr
    rds = DATA/'Full_Patient_metadata.rds'
    if not rds.exists(): rds.write_bytes(gzip.decompress((DATA/'Full_Patient_metadata.rds.gz').read_bytes()))
    META = pyreadr.read_r(str(rds))[None]
print('metadata:', META.shape, '| samples:', META['orig.ident'].nunique(), '| patients:', META.PatientID.nunique())

AVAILABLE = set(META['orig.ident']) if SMOKE_TEST else set(manifest) & set(META['orig.ident'])   # healthy donors / PT14 have metadata but no deposited matrices
ANN = pd.DataFrame(index=META.index)
ANN['sample'] = META['orig.ident'].astype(str); ANN['patient'] = META.PatientID.astype(str); ANN['timepoint'] = META.Timepoint.astype(str)
ANN['stage'] = ANN.timepoint.map(STAGE_OF); assert ANN.stage.notna().all()
ANN['geno'] = META.Classification.astype(str)                      # MUT / WT / ND / NoCall  (per-cell NPM1 read evidence)
ANN['CellType'] = META.main_hpca.astype(str)                       # coarse SingleR label shipped by the authors
ANN['y'] = ANN.geno.map({'MUT': 1.0, 'WT': 0.0})                   # ND / NoCall -> unlabeled (still used, unlabeled, for adaptation)
ANN['prog'] = META.main_hpca.isin(PROG_HPCA)
ANN = ANN[ANN['sample'].isin(AVAILABLE)].copy(); ANN['Cell'] = ANN.index
ANN.to_csv(OUT/'00_annotation_cells.csv.gz', index=False)
print('cells with matrices:', len(ANN), '| patients:', sorted(ANN.patient.unique()))

P = ANN[ANN.prog & ANN.y.notna()]
print('\nLabelled progenitor-compartment cells (MUT = mutant NPM1 reads; WT = wild-type reads only):')
display(P.groupby(['stage', 'geno']).size().unstack(fill_value=0))
print('STAGE CONFOUND: fraction malignant among labelled progenitors, by stage')
display(P.groupby('stage').y.mean().round(3).to_frame('fraction_MUT'))
print('Probes trained naively could learn "post-treatment => normal". Cell 10 therefore weights every stage x label group equally.')

dx = ANN[(ANN.stage == 'dx') & ANN.y.notna()]
wt_dx = dx.groupby('patient').y.apply(lambda s: 1 - s.mean()).round(3)
print('\nLABEL-NOISE CHECK: fraction of genotype-called DIAGNOSIS cells that are WT-only, per patient')
print('(near 0 => leukemic cells almost never look WT-only, so WT calls after treatment are mostly genuinely normal)')
display(wt_dx.to_frame('WT_fraction_at_diagnosis').T)

pn = P[(P.stage == 'post') & (P.y == 0)]; pm = P[(P.stage == 'post') & (P.y == 1)]
gate = dict(patients_total=int(ANN.patient.nunique()),
            post_normal_cells=len(pn), post_normal_patients_ge_min=int((pn.groupby('patient').size() >= MIN_CELLS_PATIENT_STRATUM).sum()),
            post_normal_top_patient_share=float(pn.patient.value_counts(normalize=True).iloc[0]) if len(pn) else np.nan,
            post_malignant_cells=len(pm), post_malignant_patients_ge_min=int((pm.groupby('patient').size() >= MIN_CELLS_PATIENT_STRATUM).sum()),
            median_WT_fraction_at_dx=float(wt_dx.median()))
GATE = pd.Series(gate); display(GATE.to_frame('value')); GATE.to_csv(OUT/'00_feasibility_gate.csv')
need = 3 if SMOKE_TEST else 5
PRIMARY_CONFIRMATORY = gate['post_normal_patients_ge_min'] >= need and gate['post_normal_cells'] >= (50 if SMOKE_TEST else 100) and gate['post_malignant_patients_ge_min'] >= 3
print('\nGate met (>=5 patients and >=100 cells of post-treatment normal progenitors; >=3 patients with post-treatment malignant progenitors):', PRIMARY_CONFIRMATORY)
print('''
LIMITS TO KEEP IN MIND
 * Labels are single-locus: NPM1 mutant vs wild-type reads. Only NPM1-mutated AML (10 patients) is covered; findings do not transfer to other genotypes.
 * WT-only is a weak normal label in principle (heterozygous leukemic cells can drop the mutant allele). The diagnosis WT fraction above bounds that error;
   it is NOT small in every patient (e.g. PT06/PT07 contain real residual normal cells at diagnosis).
 * "Progenitor" is a coarse SingleR label (GMP/CMP/MEP/HSC), not a curated gate.
 * Healthy-donor cells exist in the metadata but their count matrices are not deposited, so there is no healthy-marrow stratum.
 * Day 14/30 samples include persistent disease, not only remission marrow. This is not a clinical MRD assay.''')

In [ ]:
#@title 5 · Assemble the count matrix (cells x genes, raw UMIs) for labelled + unlabelled cells
import anndata as ad
H5 = DATA/'naldini_counts.h5ad'
N_SYN_GENES = 600

def read_10x(sample):
    """barcodes/features/matrix -> (cell_ids, gene_symbols, csr cells x genes). Fast pandas parser for the .mtx."""
    bc = pd.read_csv(DATA/'mtx'/f'{sample}_barcodes.tsv.gz', sep='\t', header=None)[0].astype(str).str.replace(r'-1$', '', regex=True)
    ft = pd.read_csv(DATA/'mtx'/f'{sample}_features.tsv.gz', sep='\t', header=None)
    with gzip.open(DATA/'mtx'/f'{sample}_matrix.mtx.gz', 'rb') as fh:
        line = fh.readline()
        while line.startswith(b'%'): line = fh.readline()
        nr, nc, nnz = map(int, line.split())
        df = pd.read_csv(fh, sep=' ', header=None, dtype=np.int32, engine='c')
    M = sp.csr_matrix((df[2].to_numpy(np.float32), (df[1].to_numpy() - 1, df[0].to_numpy() - 1)), shape=(nc, nr))
    return (sample + '_' + bc).tolist(), ft[1].astype(str).to_numpy(), M

if H5.exists() and not SMOKE_TEST:
    adata = ad.read_h5ad(H5)
else:
    if SMOKE_TEST:
        r = np.random.default_rng(1); genes = np.array([f'G{i:04d}' for i in range(N_SYN_GENES)]); base = r.gamma(0.5, 1.0, N_SYN_GENES) * 0.8
        mat = np.zeros((len(ANN), N_SYN_GENES), np.float32)
        for i, (mal, pg) in enumerate(zip(ANN.y.values, ANN.prog.values)):
            lam = base.copy()
            if mal == 1: lam[:60] *= 4
            if pg: lam[60:120] *= 3
            mat[i] = r.poisson(lam)
        adata = ad.AnnData(sp.csr_matrix(mat), obs=ANN.copy(), var=pd.DataFrame(index=genes))
    else:
        blocks, gene_ref = [], None
        for s in tqdm(sorted(AVAILABLE), desc='reading matrices'):
            cells, symbols, M = read_10x(s)
            if gene_ref is None: gene_ref = symbols
            assert (symbols == gene_ref).all(), f'feature order differs in {s}'
            keep = [i for i, c in enumerate(cells) if c in ANN.index]
            blocks.append((np.array(cells)[keep], M[keep]))
        cells = np.concatenate([b[0] for b in blocks]); X = sp.vstack([b[1] for b in blocks], format='csr')
        codes, uniq = pd.factorize(pd.Index(gene_ref))                       # duplicated symbols (e.g. 24 here) are summed, not suffixed
        A = sp.csr_matrix((np.ones(len(codes), np.float32), (np.arange(len(codes)), codes)), shape=(len(codes), len(uniq)))
        X = (X @ A).tocsr(); obs_ = ANN.loc[cells].copy()
        ok = np.asarray(X.sum(1)).ravel() > 0
        adata = ad.AnnData(X[ok], obs=obs_[ok].copy(), var=pd.DataFrame(index=np.asarray(uniq, dtype=str)))
        print('cells in metadata without a matrix column:', len(ANN) - adata.n_obs)
        adata.write_h5ad(H5)
assert adata.obs_names.is_unique
assert np.allclose(adata.X.data, np.rint(adata.X.data)), 'counts are not integers'
print(adata)
obs = adata.obs.copy(); obs['cell'] = obs.index
PROG_MASK = (obs.prog & obs.y.notna()).to_numpy(); PROG_IDX = np.flatnonzero(PROG_MASK)
print('labelled progenitor cells for evaluation:', len(PROG_IDX), 'of', adata.n_obs)

In [ ]:
#@title 6 · scGPT: checkpoint, embedding function, and continued-training (adaptation) loop
import importlib, types, scanpy as sc
import scgpt
from scgpt.model import TransformerModel
from scgpt.data_collator import DataCollator
from scgpt.loss import masked_mse_loss
from scgpt.utils import load_pretrained
from torch.utils.data import DataLoader, RandomSampler

class JSONGeneVocab:
    """Minimal vocab (torchtext-free) that keeps the checkpoint's EXACT token ids (same trick as your zero-shot notebook)."""
    def __init__(self, token_to_id):
        self._stoi = {str(k): int(v) for k, v in token_to_id.items()}
        if sorted(self._stoi.values()) != list(range(len(self._stoi))): raise ValueError('vocab ids must be consecutive')
    def __len__(self): return len(self._stoi)
    def __contains__(self, t): return t in self._stoi
    def __getitem__(self, t): return self._stoi[t]
    def append_token(self, t): self._stoi[t] = len(self._stoi)
    def get_stoi(self): return dict(self._stoi)

SCGPT_DIR = MODEL/'scGPT-human'
def prepare_checkpoint():
    SCGPT_DIR.mkdir(exist_ok=True)
    if SMOKE_TEST:      # tiny random "pretrained" checkpoint written in the real file format
        genes = [f'G{i:04d}' for i in range(N_SYN_GENES)]
        stoi = {t: i for i, t in enumerate(['<pad>', '<cls>', '<eoc>'] + genes)}
        cfg = dict(embsize=32, nheads=2, d_hid=32, nlayers=2, n_layers_cls=1, dropout=0.1, pad_token='<pad>', pad_value=-2, mask_value=-1)
        (SCGPT_DIR/'vocab.json').write_text(json.dumps(stoi)); (SCGPT_DIR/'args.json').write_text(json.dumps(cfg))
        torch.manual_seed(0); m, _ = build_scgpt(SCGPT_DIR, pretrained=False, device='cpu')
        torch.save(m.state_dict(), SCGPT_DIR/'best_model.pt')
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id='wanglab/scGPT-human', repo_type='model', allow_patterns=['args.json', 'vocab.json', 'best_model.pt'], local_dir=str(SCGPT_DIR))
    assert all((SCGPT_DIR/f).exists() for f in ('args.json', 'vocab.json', 'best_model.pt'))

def build_scgpt(model_dir, pretrained=True, device=DEVICE):
    """Mirror of scgpt.tasks.embed_data's model construction (non-flash attention), optionally with random init."""
    model_dir = Path(model_dir)
    vocab = JSONGeneVocab(json.loads((model_dir/'vocab.json').read_text()))
    for t in ['<pad>', '<cls>', '<eoc>']:
        if t not in vocab: vocab.append_token(t)
    cfg = json.loads((model_dir/'args.json').read_text())
    model = TransformerModel(ntoken=len(vocab), d_model=cfg['embsize'], nhead=cfg['nheads'], d_hid=cfg['d_hid'], nlayers=cfg['nlayers'],
                             nlayers_cls=cfg['n_layers_cls'], n_cls=1, vocab=vocab, dropout=cfg['dropout'], pad_token=cfg['pad_token'],
                             pad_value=cfg['pad_value'], do_mvc=True, do_dab=False, use_batch_labels=False, domain_spec_batchnorm=False,
                             explicit_zero_prob=False, use_fast_transformer=False, fast_transformer_backend='flash', pre_norm=False)
    if pretrained:
        sd = torch.load(model_dir/'best_model.pt', map_location='cpu')
        load_pretrained(model, sd, verbose=False)
        sd2 = {k.replace('Wqkv.', 'in_proj_'): v for k, v in sd.items()}
        ms = model.state_dict(); ok = sum(k in sd2 and sd2[k].shape == v.shape for k, v in ms.items())
        print(f'pretrained tensors loaded: {ok}/{len(ms)}')
        assert ok >= 0.9*len(ms), 'most tensors failed to load; do not interpret results'
    cfg.setdefault('mask_value', -1)
    return model.to(device), (vocab, cfg)

class CellDS(torch.utils.data.Dataset):
    """Sparse raw counts -> {<cls> + expressed genes (vocab ids), values}; binning/masking happen in scGPT's DataCollator."""
    def __init__(self, X, gene_ids, cls_id, pad_value):
        self.X, self.g, self.cls, self.pv = X.tocsr(), gene_ids, cls_id, pad_value
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        s, e = self.X.indptr[i], self.X.indptr[i+1]
        genes = np.insert(self.g[self.X.indices[s:e]], 0, self.cls); vals = np.insert(self.X.data[s:e].astype(np.float32), 0, self.pv)
        return dict(id=i, genes=torch.from_numpy(genes).long(), expressions=torch.from_numpy(vals))

def vocab_map(var_names, vocab):
    ids = np.array([vocab[g] if g in vocab else -1 for g in var_names]); return ids, ids >= 0

@torch.no_grad()
def embed_cells(model, X, gene_ids, vocab, cfg, batch_size=32, seed=0):
    """Cell embedding = <cls> state, L2-normalised (identical recipe to scgpt.tasks.embed_data)."""
    torch.manual_seed(seed); model.eval(); pad_id = vocab[cfg['pad_token']]
    ds = CellDS(X, gene_ids, vocab['<cls>'], cfg['pad_value'])
    col = DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=False, do_binning=True,
                       max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=col, num_workers=NUM_WORKERS)
    out, n = np.zeros((len(ds), cfg['embsize']), np.float32), 0
    dev = next(model.parameters()).device
    for b in tqdm(dl, desc='embedding', leave=False):
        g = b['gene'].to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
            h = model._encode(g, b['expr'].to(dev), src_key_padding_mask=g.eq(pad_id))[:, 0, :]
        out[n:n+len(h)] = h.float().cpu().numpy(); n += len(h)
    assert np.isfinite(out).all(), 'non-finite embeddings'
    return out / np.linalg.norm(out, axis=1, keepdims=True).clip(1e-8)

class RandMaskCollator:
    """scGPT pretraining drew the mask ratio from {0.25,0.5,0.75}; do the same per batch."""
    def __init__(self, base, ratios): self.base, self.ratios = base, ratios
    def __call__(self, ex):
        self.base.mlm_probability = float(self.ratios[int(torch.randint(len(self.ratios), (1,)))]); return self.base(ex)

def _loss(model, b, dev, pad_id, mask_value):
    g, tgt, masked = b['gene'].to(dev), b['expr'].to(dev), b['masked_expr'].to(dev)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
        o = model(g, masked, src_key_padding_mask=g.eq(pad_id), MVC=True)
    pos = masked.eq(mask_value)
    return masked_mse_loss(o['mlm_output'].float(), tgt.float(), pos), masked_mse_loss(o['mvc_output'].float(), tgt.float(), pos)

@torch.no_grad()
def heldout_masked_loss(model, ds, vocab, cfg, n_batches=8, batch=8):
    """Fixed-seed masked-value loss on held-out cells of the TRAINING patients: a sanity check that adaptation learns."""
    torch.manual_seed(1234); model.eval(); dev = next(model.parameters()).device; pad_id = vocab[cfg['pad_token']]
    col = RandMaskCollator(DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=True, do_binning=True,
                           mask_value=cfg['mask_value'], max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1), (0.5,))
    dl = DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=col); tot = []
    for i, b in enumerate(dl):
        if i >= n_batches: break
        a, c = _loss(model, b, dev, pad_id, cfg['mask_value']); tot.append((a + c).item())
    return float(np.mean(tot))

def adapt_scgpt(model, X, gene_ids, vocab, cfg, steps, lr, seed, label):
    """Continued masked-value training (GEP + GEPC) on UNLABELED cells. Patient labels/malignancy are never used."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); dev = next(model.parameters()).device; pad_id = vocab[cfg['pad_token']]
    perm = rng.permutation(X.shape[0]); n_val = min(128, max(8, X.shape[0]//20))
    val_ds = CellDS(X[perm[:n_val]], gene_ids, vocab['<cls>'], cfg['pad_value'])
    tr_ds = CellDS(X[perm[n_val:]], gene_ids, vocab['<cls>'], cfg['pad_value'])
    base = DataCollator(do_padding=True, pad_token_id=pad_id, pad_value=cfg['pad_value'], do_mlm=True, do_binning=True,
                        mask_value=cfg['mask_value'], max_length=MAX_LEN, sampling=True, keep_first_n_tokens=1)
    gen = torch.Generator().manual_seed(seed)
    dl = DataLoader(tr_ds, batch_size=ADAPT_BATCH, sampler=RandomSampler(tr_ds, replacement=True, num_samples=steps*ADAPT_ACCUM*ADAPT_BATCH, generator=gen),
                    collate_fn=RandMaskCollator(base, MASK_RATIOS), num_workers=NUM_WORKERS, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    warm = max(1, int(0.05*steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.1 + 0.9*0.5*(1+math.cos(math.pi*(s-warm)/max(1, steps-warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=dev.type == 'cuda')
    l0 = heldout_masked_loss(model, val_ds, vocab, cfg); hist = []
    model.train(); step, micro, t0 = 0, 0, time.time()
    for b in tqdm(dl, total=steps*ADAPT_ACCUM, desc=f'adapt[{label}]', leave=False):
        a, c = _loss(model, b, dev, pad_id, cfg['mask_value'])
        scaler.scale((a + c)/ADAPT_ACCUM).backward(); micro += 1
        if micro % ADAPT_ACCUM == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            hist.append((a + c).item())
    l1 = heldout_masked_loss(model, val_ds, vocab, cfg)
    print(f'  [{label}] held-out masked loss (training patients): {l0:.4f} -> {l1:.4f} | {steps} steps, {(time.time()-t0)/60:.1f} min')
    model.eval(); return dict(label=label, loss_start=l0, loss_end=l1, train_loss=hist)

prepare_checkpoint()
model0, (VOCAB, CFG) = build_scgpt(SCGPT_DIR, pretrained=True)
GENE_IDS_ALL, IN_VOCAB = vocab_map(adata.var_names.astype(str), VOCAB)
print(f'scGPT vocabulary covers {IN_VOCAB.sum():,}/{len(IN_VOCAB):,} genes ({IN_VOCAB.mean():.1%})')
assert IN_VOCAB.mean() > 0.5 if not SMOKE_TEST else True
X_V = adata.X[:, np.flatnonzero(IN_VOCAB)].tocsr(); GENE_IDS = GENE_IDS_ALL[IN_VOCAB]   # scGPT-visible genes only

In [ ]:
#@title 7 · Patient-grouped folds (greedy balance so each fold holds post-treatment normal progenitors)
groups = obs.groupby('patient').apply(lambda d: pd.Series(dict(
    n_prog=int(d.index.isin(obs.index[PROG_IDX]).sum()),
    n_post_norm=int(((d.stage == 'post') & (d.y == 0) & d.prog).sum()),
    n_post_mal=int(((d.stage == 'post') & (d.y == 1) & d.prog).sum())))).reset_index()
groups['w'] = groups.n_prog + 10*groups.n_post_norm + 10*groups.n_post_mal
fold_w = np.zeros(N_FOLDS); FOLD_OF = {}
for _, r in groups.sort_values('w', ascending=False).iterrows():
    k = int(np.argmin(fold_w)); FOLD_OF[r.patient] = k; fold_w[k] += r.w
groups['fold'] = groups.patient.map(FOLD_OF); display(groups.sort_values(['fold', 'patient']))
FOLDS = [sorted(p for p, k in FOLD_OF.items() if k == f) for f in range(N_FOLDS)]
groups.to_csv(OUT/'01_folds.csv', index=False)

In [ ]:
#@title 8 · Original (frozen) scGPT embeddings of every labelled progenitor cell
f = CACHE/'emb_scGPT_zeroshot.npy'
if f.exists(): Z0 = np.load(f)
else:
    Z0 = embed_cells(model0, X_V[PROG_IDX], GENE_IDS, VOCAB, CFG, seed=SEED); np.save(f, Z0)
print('zero-shot embeddings', Z0.shape)
del model0
if DEVICE == 'cuda': torch.cuda.empty_cache()

In [ ]:
#@title 9 · Per-fold representations fitted on TRAINING patients only: PCA, scVI, AML-adapted scGPT, from-scratch scGPT
from sklearn.decomposition import PCA
SCVI_EPOCHS = 5 if SMOKE_TEST else 40
REPS = ['PCA'] + (['scVI'] if RUN_SCVI else []) + ['scGPT_zeroshot', 'scGPT_adapted'] + (['scGPT_scratch'] if RUN_SCRATCH else [])
EMB = {r: {} for r in REPS}; ADAPT_LOG = {}

def log_cp10k(X):
    lib = np.asarray(X.sum(1)).ravel().clip(1); Y = sp.diags(1e4/lib) @ X; Y = Y.tocsr(); Y.data = np.log1p(Y.data); return Y

def fit_hvg_pca(train_rows, n_hvg=3000, n_comp=50, seed=SEED):
    r = np.random.default_rng(seed); rows = r.choice(train_rows, size=min(len(train_rows), 20000), replace=False)
    Y = log_cp10k(adata.X[rows]); mu = np.asarray(Y.mean(0)).ravel(); var = np.asarray(Y.power(2).mean(0)).ravel() - mu**2
    hvg = np.argsort(var)[-min(n_hvg, Y.shape[1]):]
    D = Y[:, hvg].toarray(); center = D.mean(0)
    pca = PCA(n_components=min(n_comp, D.shape[1]-1, D.shape[0]-1), random_state=seed).fit(D - center)
    return hvg, center, pca

def adapt_corpus(train_rows, seed):
    r = np.random.default_rng(seed); keep = []
    for s, idx in obs.iloc[train_rows].groupby('sample').indices.items():
        keep.extend(train_rows[idx][r.permutation(len(idx))[:MAX_ADAPT_CELLS_PER_SAMPLE]])
    return np.sort(np.array(keep))

def run_scvi(train_rows, hvg, seed):
    import scvi
    scvi.settings.seed = seed
    r = np.random.default_rng(seed); rows = np.sort(r.choice(train_rows, size=min(len(train_rows), 30000), replace=False))
    a_tr = ad.AnnData(adata.X[rows][:, hvg].astype(np.float32)); scvi.model.SCVI.setup_anndata(a_tr)
    m = scvi.model.SCVI(a_tr, n_latent=30); m.train(max_epochs=SCVI_EPOCHS, early_stopping=False)
    a_p = ad.AnnData(adata.X[PROG_IDX][:, hvg].astype(np.float32))
    return np.asarray(m.get_latent_representation(a_p), dtype=np.float32)

for k in range(N_FOLDS):
    test = FOLDS[k]; train_rows = np.flatnonzero(~obs.patient.isin(test).to_numpy())
    print(f'\n=== fold {k}: held-out patients {test} | {len(train_rows):,} training cells ===')
    EMB['scGPT_zeroshot'][k] = Z0
    # --- PCA (+ shared HVG set for scVI) ---
    f = CACHE/f'fold{k}_PCA.npy'
    hvg, center, pca = fit_hvg_pca(train_rows)
    if f.exists(): EMB['PCA'][k] = np.load(f)
    else:
        D = log_cp10k(adata.X[PROG_IDX])[:, hvg].toarray() - center; EMB['PCA'][k] = pca.transform(D).astype(np.float32); np.save(f, EMB['PCA'][k])
    # --- scVI ---
    if RUN_SCVI:
        f = CACHE/f'fold{k}_scVI.npy'
        if f.exists(): EMB['scVI'][k] = np.load(f)
        else:
            try: EMB['scVI'][k] = run_scvi(train_rows, hvg, SEED+k); np.save(f, EMB['scVI'][k])
            except Exception as e:
                print('  scVI failed -> dropped from comparison:', type(e).__name__, str(e)[:200]); RUN_SCVI = False; REPS.remove('scVI'); EMB.pop('scVI', None)
    # --- scGPT adapted / scratch: unlabeled masked-value training on TRAINING patients' cells only ---
    for rep, pre, lr in [('scGPT_adapted', True, ADAPT_LR)] + ([('scGPT_scratch', False, SCRATCH_LR)] if RUN_SCRATCH else []):
        f = CACHE/f'fold{k}_{rep}.npy'
        if f.exists(): EMB[rep][k] = np.load(f); continue
        corpus = adapt_corpus(train_rows, SEED+k)
        torch.manual_seed(SEED+k); m, _ = build_scgpt(SCGPT_DIR, pretrained=pre)
        ADAPT_LOG[f'{rep}_fold{k}'] = adapt_scgpt(m, X_V[corpus], GENE_IDS, VOCAB, CFG, ADAPT_STEPS, lr, SEED+k, f'{rep}/fold{k}')
        EMB[rep][k] = embed_cells(m, X_V[PROG_IDX], GENE_IDS, VOCAB, CFG, seed=SEED); np.save(f, EMB[rep][k])
        del m
        if DEVICE == 'cuda': torch.cuda.empty_cache()
json.dump({k: {a: b for a, b in v.items() if a != 'train_loss'} for k, v in ADAPT_LOG.items()}, open(OUT/'02_adaptation_log.json', 'w'), indent=1)
print('\nrepresentations ready:', REPS)

In [ ]:
#@title 10 · Identical linear probe per representation; stage-balanced; operating point fixed on TRAINING patients (nested, patient-grouped)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

po = obs.iloc[PROG_IDX].reset_index(drop=True)      # row i of every embedding == po.iloc[i]
y_all = po.y.to_numpy().astype(int); stage_all = po.stage.to_numpy()

def capped(idx, seed):
    r = np.random.default_rng(seed); keep = []
    for _, g in po.iloc[idx].groupby(['sample', 'y']).indices.items():
        keep.extend(idx[g][r.permutation(len(g))[:PROBE_MAX_PER_SAMPLE_LABEL]])
    return np.sort(np.array(keep))

def stage_weights(stage, y):
    """Every (stage, label) group gets equal total weight, so 'post-treatment => normal' cannot be a shortcut."""
    key = pd.Series(stage).astype(str) + '|' + pd.Series(y).astype(str)
    w = 1.0 / key.map(key.value_counts()).to_numpy(); return w / w.mean()

def wquantile(x, w, q):
    o = np.argsort(x); x, w = x[o], w[o]; c = (np.cumsum(w) - 0.5*w) / w.sum(); return float(np.interp(q, c, x))

def lr(C): return LogisticRegression(C=C, class_weight=None, max_iter=3000)

def probe_fold(E, k, seed=SEED):
    te = po.patient.isin(FOLDS[k]).to_numpy(); tr_idx = capped(np.flatnonzero(~te), seed+k); te_idx = np.flatnonzero(te)
    sc = StandardScaler().fit(E[tr_idx]); A = sc.transform(E[tr_idx]); yt = y_all[tr_idx]; g = po.patient.to_numpy()[tr_idx]
    w = stage_weights(stage_all[tr_idx], yt); best = (-1, None, None)
    for C in C_GRID:                                   # inner patient-grouped CV picks C and yields out-of-fold training scores
        oof = np.full(len(A), np.nan)
        for a, b in GroupKFold(n_splits=3).split(A, yt, g):
            if len(np.unique(yt[a])) < 2: continue
            oof[b] = lr(C).fit(A[a], yt[a], sample_weight=w[a]).decision_function(A[b])
        ok = ~np.isnan(oof)
        auc = roc_auc_score(yt[ok], oof[ok], sample_weight=w[ok]) if ok.sum() and len(np.unique(yt[ok])) == 2 else -1
        if auc > best[0]: best = (auc, C, oof)
    auc_in, C, oof = best
    mal = (yt == 1) & ~np.isnan(oof)
    thr = wquantile(oof[mal], w[mal], 1 - TARGET_TPR)    # TARGET_TPR on training malignant cells, stage-balanced
    s = lr(C).fit(A, yt, sample_weight=w).decision_function(sc.transform(E[te_idx]))
    return pd.DataFrame(dict(row=te_idx, rep=None, fold=k, score=s, call=(s >= thr).astype(int), C=C, inner_auc=auc_in, thr=thr))

parts = []
for rep in REPS:
    for k in range(N_FOLDS):
        d = probe_fold(EMB[rep][k], k); d['rep'] = rep; parts.append(d)
PRED = pd.concat(parts, ignore_index=True)
meta = po[['cell', 'patient', 'sample', 'stage', 'y', 'geno', 'CellType']].copy(); meta['row'] = np.arange(len(po))
PRED = PRED.merge(meta, on='row'); PRED.to_csv(OUT/'03_cell_level_predictions.csv.gz', index=False)
display(PRED.groupby('rep')[['inner_auc']].mean().round(3).rename(columns={'inner_auc': 'mean inner-CV weighted AUROC (training patients)'}))

In [ ]:
#@title 11 · False-positive rate at the fixed operating point, with patient-cluster bootstrap and pre-specified contrasts
STRATA = {   # false-positive strata (true label = WT-only, i.e. non-malignant)
    'post_normal':    lambda d: (d.stage == 'post') & (d.y == 0),
    'dx_normal':      lambda d: (d.stage == 'dx') & (d.y == 0),
    'relapse_normal': lambda d: (d.stage == 'relapse') & (d.y == 0)}
SENS = {     # sensitivity strata (true label = MUT)
    'mal_all':     lambda d: d.y == 1,
    'mal_post':    lambda d: (d.stage == 'post') & (d.y == 1),
    'mal_relapse': lambda d: (d.stage == 'relapse') & (d.y == 1)}
ALLS = {**STRATA, **SENS}
rng = np.random.default_rng(SEED)

def patient_rates(rep, name):
    d = PRED[PRED.rep == rep]; d = d[ALLS[name](d)]
    g = d.groupby('patient').call.agg(['mean', 'size']); return g[g['size'] >= MIN_CELLS_PATIENT_STRATUM]['mean']

rows, BOOT = [], {}
for name in ALLS:
    pats = sorted(set.intersection(*[set(patient_rates(r, name).index) for r in REPS]))
    if not pats: continue
    idx = rng.integers(0, len(pats), size=(N_BOOT, len(pats)))          # SAME patient resamples for every representation (paired)
    for rep in REPS:
        v = patient_rates(rep, name).reindex(pats).to_numpy(); b = v[idx].mean(1); BOOT[(rep, name)] = b
        d = PRED[PRED.rep == rep]; d = d[ALLS[name](d)]
        rows.append(dict(stratum=name, rep=rep, n_patients=len(pats), n_cells=int(len(d)), macro_rate=v.mean(),
                         ci_lo=np.percentile(b, 2.5), ci_hi=np.percentile(b, 97.5), pooled_rate=d.call.mean()))
RES = pd.DataFrame(rows); RES.to_csv(OUT/'04_rates_by_stratum.csv', index=False)
print('call rate by stratum.  FALSE-POSITIVE strata want LOW; SENSITIVITY strata (mal_*) should sit near', TARGET_TPR)
display(RES.pivot(index='rep', columns='stratum', values='macro_rate').reindex(REPS).round(3))
display(RES[['stratum', 'rep', 'n_patients', 'n_cells']].pivot(index='stratum', columns='rep', values='n_patients'))

# ---- pre-specified contrasts on the PRIMARY stratum (post-treatment normal progenitors) ----
def contrast(a, b, name='post_normal'):
    if (a, name) not in BOOT or (b, name) not in BOOT: return None
    d = BOOT[(a, name)] - BOOT[(b, name)]
    pt = RES[(RES.rep == a) & (RES.stratum == name)].macro_rate.iloc[0] - RES[(RES.rep == b) & (RES.stratum == name)].macro_rate.iloc[0]
    return pt, np.percentile(d, 2.5), np.percentile(d, 97.5)
def tpr_gap(a, b):
    ra, rb = (RES[(RES.rep == x) & (RES.stratum == 'mal_all')].macro_rate.iloc[0] for x in (a, b)); return ra - rb
CONTRASTS = [('scGPT_adapted', 'scGPT_zeroshot', 'C1 adaptation helps (vs original scGPT)'),
             ('scGPT_adapted', 'PCA',            'C2 beats PCA'),
             ('scGPT_adapted', 'scVI',           'C3 beats scVI')] + ([('scGPT_adapted', 'scGPT_scratch', 'C4 pretraining matters (vs scratch)')] if RUN_SCRATCH else [])
crow = []
for a, b, lab in CONTRASTS:
    c = contrast(a, b)
    if c is None: continue
    crow.append(dict(contrast=lab, FPR_difference=c[0], ci_lo=c[1], ci_hi=c[2], sensitivity_gap=tpr_gap(a, b),
                     FPR_lower_CI_excludes_0=c[2] < 0, sensitivity_matched_within_5pts=abs(tpr_gap(a, b)) <= 0.05))
CT = pd.DataFrame(crow); CT.to_csv(OUT/'05_primary_contrasts.csv', index=False); display(CT.round(3))

# ---- secondary: discrimination WITHIN post-treatment progenitors (MUT vs WT), patient-cluster bootstrap ----
def post_cells(rep): d = PRED[(PRED.rep == rep) & (PRED.stage == 'post')]; return d
auc_rows = []
pats_auc = sorted(set(post_cells(REPS[0]).patient))
rng2 = np.random.default_rng(SEED + 1); idx2 = rng2.integers(0, len(pats_auc), size=(min(N_BOOT, 1000), len(pats_auc)))
for rep in REPS:
    d = post_cells(rep); by = {p: g for p, g in d.groupby('patient')}
    pt = roc_auc_score(d.y, d.score) if d.y.nunique() == 2 else np.nan
    bs = []
    for row in idx2:
        g = pd.concat([by[pats_auc[i]] for i in row]); bs.append(roc_auc_score(g.y, g.score) if g.y.nunique() == 2 else np.nan)
    auc_rows.append(dict(rep=rep, auroc_within_post=pt, ci_lo=np.nanpercentile(bs, 2.5), ci_hi=np.nanpercentile(bs, 97.5),
                         n_cells=len(d), n_malignant=int(d.y.sum()), n_patients_with_both=int(d.groupby('patient').y.nunique().eq(2).sum())))
AUC = pd.DataFrame(auc_rows); AUC.to_csv(OUT/'05b_auroc_within_post.csv', index=False)
print('Secondary: MUT-vs-WT AUROC among post-treatment progenitors (pooled over folds; fold scores are not perfectly co-calibrated):')
display(AUC.round(3))

In [ ]:
#@title 12 · Figures: false-positive rate by representation; per-patient view
import matplotlib.pyplot as plt
OKABE = {'PCA': '#56B4E9', 'scVI': '#009E73', 'scGPT_zeroshot': '#999999', 'scGPT_adapted': '#D55E00', 'scGPT_scratch': '#CC79A7'}
LABEL = {'PCA': 'HVG PCA', 'scVI': 'scVI', 'scGPT_zeroshot': 'scGPT original', 'scGPT_adapted': 'scGPT AML-adapted', 'scGPT_scratch': 'scGPT from scratch'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

TITLES = {'post_normal': 'Post-treatment (D14/D30) normal progenitors\n(PRIMARY)', 'dx_normal': 'Diagnosis normal progenitors', 'relapse_normal': 'Relapse normal progenitors'}
present = [s_ for s_ in TITLES if s_ in set(RES.stratum)]
fig, axes = plt.subplots(1, len(present), figsize=(4.3*len(present), 4.2), sharey=True, squeeze=False); axes = axes[0]
for ax, name in zip(axes, present):
    title = TITLES[name]
    sub = RES[RES.stratum == name].set_index('rep').reindex(REPS)
    for i, rep in enumerate(REPS):
        r = sub.loc[rep]; ax.errorbar(i, r.macro_rate, yerr=[[r.macro_rate - r.ci_lo], [r.ci_hi - r.macro_rate]], fmt='o', ms=8,
                                      color=OKABE[rep], mec='white', mew=1.5, capsize=3, lw=1.5)
    ax.set_xticks(range(len(REPS))); ax.set_xticklabels([LABEL[r] for r in REPS], rotation=30, ha='right')
    ax.set_title(f'{title}\nn={int(sub.n_patients.iloc[0])} patients, {int(sub.n_cells.iloc[0])} cells', fontsize=10, loc='left')
axes[0].set_ylabel(f'False-positive rate at ~{int(TARGET_TPR*100)}% training sensitivity\n(patient-macro mean, 95% patient bootstrap CI)')
axes[0].set_ylim(0, 1); fig.tight_layout(); fig.savefig(FIG/'01_fpr_by_stratum.png', dpi=200); plt.show()

pp = PRED[PRED.stage.eq('post') & PRED.y.eq(0)].groupby(['patient', 'rep']).call.agg(['mean', 'size']).reset_index()
pp = pp[pp['size'] >= MIN_CELLS_PATIENT_STRATUM]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for p, g in pp.groupby('patient'):
    g = g.set_index('rep').reindex(REPS); ax.plot(range(len(REPS)), g['mean'], color='#BBBBBB', lw=1, zorder=1)
    ax.text(len(REPS)-0.9, g['mean'].iloc[-1], f'{p} (n={int(g["size"].iloc[0])})', fontsize=7, va='center', color='#555555')
for i, rep in enumerate(REPS): ax.scatter([i]*len(pp[pp.rep == rep]), pp[pp.rep == rep]['mean'], color=OKABE[rep], s=40, zorder=2, edgecolor='white')
ax.set_xticks(range(len(REPS))); ax.set_xticklabels([LABEL[r] for r in REPS], rotation=30, ha='right'); ax.set_ylim(-0.02, 1.02)
ax.set_ylabel('Fraction of post-treatment normal progenitors\ncalled malignant'); ax.set_title('Each line is one patient: is the effect shared or driven by one person?', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG/'02_post_normal_per_patient.png', dpi=200); plt.show()

In [ ]:
#@title 13 · Pre-specified decision rule, save outputs, optional all-patient adapted checkpoint
n_pat = int(RES[(RES.stratum == 'post_normal')].n_patients.iloc[0]); n_cell = int(RES[(RES.stratum == 'post_normal')].n_cells.iloc[0])
crit = {r.contrast: bool(r.FPR_lower_CI_excludes_0 and r.sensitivity_matched_within_5pts) for r in CT.itertuples()}
print('Per-contrast pass = FPR difference CI entirely below 0 AND sensitivity within 5 points of the comparator:')
for k_, v in crit.items(): print(f'  {"PASS" if v else "fail"}  {k_}')
core = [v for k_, v in crit.items() if k_.startswith(('C1', 'C2', 'C3'))]
if len(core) < 3:
    verdict = 'INCOMPLETE: a required comparator (scVI) is missing, so the pre-specified decision rule cannot be applied. Fix scVI and re-run.'
elif not PRIMARY_CONFIRMATORY:
    verdict = (f'UNDERPOWERED / EXPLORATORY: primary stratum has {n_pat} patients, {n_cell} cells (gate not met). '
               'Even if C1-C3 pass, this supports only a pilot signal, not a model-development claim.')
elif all(core):
    verdict = 'GO: adaptation lowered post-treatment false positives vs original scGPT, PCA and scVI at matched sensitivity. Next: replicate in an independent cohort (other genotypes, ideally donor/recipient-labelled post-transplant marrow); this cohort is NPM1-mutated AML only.'
else:
    verdict = 'NO-GO: adaptation did not beat all of original scGPT, PCA and scVI on the primary endpoint. Per the decision rule fixed in this notebook before any real result, abandon the model-development claim.'
print('\nVERDICT:', verdict)
(OUT/'README_RESULTS.md').write_text(f'''# AML-adapted scGPT false-positive benchmark (Naldini GSE185991)

**Verdict:** {verdict}

Primary stratum: {n_pat} patients / {n_cell} cells. Labels are single-locus NPM1 mutant/wild-type read calls (Naldini et al. 2023, NPM1-mutated AML only; healthy-donor matrices not deposited).\nGate and label-noise check: 00_feasibility_gate.csv. Not a clinical MRD assay; no MRD sensitivity claim.

Files: 04_rates_by_stratum.csv, 05_primary_contrasts.csv, 03_cell_level_predictions.csv.gz, 02_adaptation_log.json, figures/.
''')
print((OUT/'README_RESULTS.md').read_text())

if TRAIN_FINAL_MODEL:
    f = MODEL/'scGPT_AML_adapted_allpatients.pt'
    if not f.exists():
        corpus = adapt_corpus(np.arange(adata.n_obs), SEED+999)
        torch.manual_seed(SEED); m, _ = build_scgpt(SCGPT_DIR, pretrained=True)
        adapt_scgpt(m, X_V[corpus], GENE_IDS, VOCAB, CFG, ADAPT_STEPS, ADAPT_LR, SEED+999, 'final/all-patients')
        torch.save(m.state_dict(), f)
    print('all-patient adapted checkpoint (for the seven-patient follow-up):', f)
    print('NOTE: this model has seen every patient in this cohort. Never evaluate it on this cohort; use it only on independent cohorts.')